In [5]:
import mediapipe as mp
import cv2
import numpy as np
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
from mediapipe.tasks.python.vision.drawing_utils import DrawingSpec
import pydirectinput
import pyautogui
import keyboard

In [6]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options2 = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='./hand_landmarker.task'),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    min_hand_presence_confidence=0.5)

In [7]:
sc_with = pyautogui.size().width
sc_height = pyautogui.size().height

print(sc_with, sc_height)

1920 1080


In [8]:
def is_hand_closed(landmarks, threshold=0.05):
    """
    Detect if hand is closed by checking distance from fingertips to palm.
    landmarks: list of 21 hand landmark points
    threshold: distance threshold (0-1, where 1 is full hand width)
    Returns: True if hand is closed, False if open
    """
    # Palm center (wrist landmark)
    palm = landmarks[0]
    
    # Fingertips indices: thumb(4), index(8), middle(12), ring(16), pinky(20)
    fingertip_indices = [4, 8, 12, 16, 20]
    
    distances = []
    for idx in fingertip_indices:
        tip = landmarks[idx]
        distance = ((tip.x - palm.x)**2 + (tip.y - palm.y)**2)**0.5
        distances.append(distance)
    
    # If average distance is less than threshold, hand is closed
    avg_distance = sum(distances) / len(distances)
    return avg_distance < threshold

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)

frame_index = 0

with HandLandmarker.create_from_options(options2) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break       
        
        img = frame.copy()
        img = cv2.flip(img, 1)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
        timestamp_ms = int(frame_index * 33.33)
        frame_index += 1
        
        detection = landmarker.detect_for_video(mp_img, timestamp_ms)
        hand_landmarks = detection.hand_landmarks        
        connections = mp.tasks.vision.HandLandmarksConnections.HAND_CONNECTIONS
        
        if len(hand_landmarks) > 0:
            x = 0
            y = 0
            
            for p in hand_landmarks[0]:
                x += p.x
                y += p.y
            
            mean_x = x / len(hand_landmarks[0])
            mean_y = y / len(hand_landmarks[0])
            
            print(mean_x, mean_y)
            
            is_closed = is_hand_closed(hand_landmarks[0], threshold=0.1)
            if is_closed:
                print("Hand is CLOSED (fist)")
            else:
                print("Hand is OPEN")
                
            pydirectinput.moveTo(round(mean_x*sc_with), round(mean_y*sc_height))
        
        for landmarks in hand_landmarks:
            drawing_utils.draw_landmarks(
                image=img_rgb,
                landmark_list = landmarks,
                connections=connections
            )
        
        output_frame = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
        
        if len(hand_landmarks) > 0:
            cx = int(mean_x * output_frame.shape[1])
            cy = int(mean_y * output_frame.shape[0])
            cv2.circle(output_frame, (cx, cy), radius=5, color=(255, 0, 0), thickness=-1)
        
        cv2.imshow('test', output_frame)
        if cv2.waitKey(1) and keyboard.is_pressed('q'):
            print('Stop!')
            break

cap.release()
cv2.destroyAllWindows()

0.3733871976534526 0.6683607924552191
Hand is OPEN
0.35624058686551596 0.670171993119376
Hand is OPEN
0.35389766309942516 0.6631791648410615
Hand is OPEN
0.3534807236421676 0.6633164428529286
Hand is OPEN
0.35337561723731814 0.6619279725211007
Hand is OPEN
0.3560518210842496 0.6628478424889701
Hand is OPEN
0.35818127507255193 0.664907109169733
Hand is OPEN
0.3598234163863318 0.6648971693856376
Hand is OPEN
0.36002405058769954 0.6665038381304059
Hand is OPEN
0.36029325070835294 0.6655936666897365
Hand is OPEN
0.36141292467003777 0.6679946467989967
Hand is OPEN
0.3616532967204139 0.6680583073979333
Hand is OPEN
0.36283108592033386 0.6686019613629296
Hand is OPEN
0.36265658055033 0.6708208265758696
Hand is OPEN
0.3635168813523792 0.6717633633386522
Hand is OPEN
0.3653474294003986 0.6732913880121141
Hand is OPEN
0.36774156774793354 0.6716061149324689
Hand is OPEN
0.38080566979589914 0.7030542067119053
Hand is OPEN
0.8853953083356222 0.8476397622199285
Hand is OPEN
0.8079698114168077 0.6154